In [1]:
# This code is based on https://github.com/AishSweety/hybrid-quantum-classical-Neural-Network-for-radar-data
# and adjusted for the experiments in the paper:
# "Hybrid Quantum Neural Networks with Amplitude Encoding: Advancing Recovery Rate Predictions".

In [1]:
from   signalgenerator import psigenerator, generateData
from   configurations  import *
import numpy           as     np

In [2]:
def create_data(n_xi, offset):
    """
    Generate complex-valued training, validation, and test datasets for two
    drone types in the X band, with a fixed SNR = 0 dB.

    Parameters
    ----------
    n_xi : int
        Number of complex samples per generated example (signal length).
    offset : bool or any
        Offset parameter forwarded to `generateData`. Its semantics depend
        on that function's implementation.

    Returns
    -------
    train_data_X : np.ndarray, shape (300, n_xi), dtype complex128
        Complex-valued training signals.
    train_data_y : np.ndarray, shape (300,)
        Class labels: 0 or 1 (drone index).
    val_data_X : np.ndarray, shape (100, n_xi), dtype complex128
        Complex-valued validation signals.
    val_data_y : np.ndarray, shape (100,)
        Class labels for validation signals.
    test_data_X : np.ndarray, shape (300, n_xi), dtype complex128
        Complex-valued test signals.
    test_data_y : np.ndarray, shape (300,)
        Class labels for test signals.
    """

    # Fix the SNR in the global X-band scenario to 0 dB.
    # NOTE: this mutates the global `scenarioXband` dictionary.
    scenarioXband["SNR"] = 0

    # Sampling frequency for X band in Hz
    Xf_s            = 10_000

    # Signal duration in seconds given the number of samples and sampling frequency
    sample_length   = n_xi / Xf_s

    # Number of samples in training, validation, and test sets
    # (total across both drone classes)
    trainset_size   = 300
    validation_size = 100
    testset_size    = 300

    # List of drone configuration dictionaries.
    # Each entry (e.g., `djimatrice300rtk`) presumably contains parameters
    # used by `psigenerator`.
    drones = [djimatrice300rtk,
              djimavicair2]

    def get_data(dataset_size, n_xi, scenario, f_s, drones, sample_length, offset):
        """
        Generate one dataset (training, validation, or test) of a given size
        for all drone classes.

        The dataset is balanced: half the samples come from each drone class.
        """

        # Allocate array for complex signals: shape (dataset_size, n_xi)
        data_X = np.zeros((dataset_size, n_xi), dtype=np.complex128)
        # Allocate array for labels: one label per signal
        data_y = np.zeros(dataset_size)

        # Number of samples per drone (balanced split)
        size   = dataset_size // 2

        # Loop over drone types (classes)
        for drone in range(len(drones)):

            # For each drone, generate `size` examples
            for x in range(size):

                # Generate parameter set `p` for the current drone and scenario.
                # `psigenerator` uses keyword arguments from both dictionaries.
                # Any randomness in `psigenerator` will drive variability in the samples.
                p = psigenerator(**dict(scenario, **drones[drone]))

                # Generate one signal realization with the given parameters.
                # `xs` are time samples; `real_ys` and `imaginary_ys` are the real
                # and imaginary parts of the baseband signal.
                xs, real_ys, imaginary_ys = generateData(
                    p, f_s, sample_length, offset=offset
                )

                # Combine real and imaginary parts into one complex-valued vector
                temp = np.array(real_ys) + 1j * np.array(imaginary_ys)

                # Compute global index for this sample within the dataset.
                # First `size` indices (0..size-1) = drone 0,
                # next `size` indices (size..2*size-1) = drone 1, etc.
                idx = size * drone + x

                # Store the complex signal and its class label
                data_X[idx] = temp
                data_y[idx] = drone

        return data_X, data_y

    # Generate training data for all drones using X-band scenario at 10 kHz
    train_data_X, train_data_y = get_data(
        trainset_size, n_xi, scenarioXband, Xf_s, drones, sample_length, offset
    )

    # Generate validation data for all drones using X-band scenario at 10 kHz
    val_data_X, val_data_y = get_data(
        validation_size, n_xi, scenarioXband, Xf_s, drones, sample_length, offset
    )

    # Generate test data for all drones using the same configuration
    test_data_X, test_data_y = get_data(
        testset_size, n_xi, scenarioXband, Xf_s, drones, sample_length, offset
    )

    def shuffle(data_X, data_y):
        """
        Shuffle samples and labels in unison to randomize ordering.
        """
        indices = np.arange(len(data_X))
        np.random.shuffle(indices)
        return data_X[indices], data_y[indices]

    # Shuffle training set
    train_data_X, train_data_y = shuffle(train_data_X, train_data_y)

    # Shuffle validation set
    val_data_X, val_data_y     = shuffle(val_data_X  , val_data_y  )

    # Shuffle test set
    test_data_X, test_data_y   = shuffle(test_data_X , test_data_y )

    return train_data_X, train_data_y, val_data_X, val_data_y, test_data_X, test_data_y


offset = False

# Example execution: generate 900-sample signals with a specific `offset` setting
train_data_X_900, train_data_y_900, \
val_data_X_900  , val_data_y_900,   \
test_data_X_900 , test_data_y_900   = create_data(n_xi   = 900   ,
                                                  offset = offset)

In [3]:
def bin_data(train_data_X, val_data_X, test_data_X, num_bins1, num_bins2):
    """
    Bin and normalize training, validation, and test signals into pooled
    features over fixed-size bins.

    Parameters
    ----------
    train_data_X : np.ndarray, shape (N_train, L)
        Training data, each row is a signal of length L.
    val_data_X : np.ndarray, shape (N_val, L)
        Validation data, each row is a signal of length L.
    test_data_X : np.ndarray, shape (N_test, L)
        Test data, each row is a signal of length L.
    num_bins1 : int
        First number of bins to use for pooling (e.g. 6).
    num_bins2 : int
        Second number of bins to use for pooling (e.g. 10).

    Returns
    -------
    train_data_X_pooled_mean_b1 : np.ndarray, shape (N_train, num_bins1)
        Normalized pooled features for training data using num_bins1.
    val_data_X_pooled_mean_b1 : np.ndarray, shape (N_val, num_bins1)
        Normalized pooled features for validation data using num_bins1.
    test_data_X_pooled_mean_b1 : np.ndarray, shape (N_test, num_bins1)
        Normalized pooled features for test data using num_bins1.
    train_data_X_pooled_mean_b2 : np.ndarray, shape (N_train, num_bins2)
        Normalized pooled features for training data using num_bins2.
    val_data_X_pooled_mean_b2 : np.ndarray, shape (N_val, num_bins2)
        Normalized pooled features for validation data using num_bins2.
    test_data_X_pooled_mean_b2 : np.ndarray, shape (N_test, num_bins2)
        Normalized pooled features for test data using num_bins2.
    """

    def _help(data, num_bins):
        """
        Helper that:
        1) splits each row into `num_bins` contiguous bins,
        2) computes the mean in each bin,
        3) scales by sqrt(bin_length),
        4) L2-normalizes the resulting per-row feature vector.
        """

        # data has shape (N_samples, L)
        # We reshape into (N_samples, num_bins, L_bin),
        # where L_bin = L / num_bins (must be an integer).
        data_reshaped    = data.reshape(data.shape[0], num_bins, -1)

        # Compute mean in each bin along the last axis (length L_bin).
        # Result: shape (N_samples, num_bins).
        # Multiply by sqrt(L_bin) to preserve energy (similar to averaging then scaling).
        pooled_mean      = data_reshaped.mean(axis=2) * np.sqrt(data_reshaped.shape[2])

        # Compute L2 norm of each row feature vector:
        # shape (N_samples, 1) so it can be broadcast in division.
        norm_mean        = np.linalg.norm(pooled_mean, axis=1, keepdims=True)

        # Normalize each row to unit L2 norm.
        # If there are zero rows, this will result in NaNs or infs; caller must handle if relevant.
        data_pooled_mean = pooled_mean / norm_mean

        return data_pooled_mean

    # Use first binning configuration (num_bins1) on training, validation, and test data
    train_data_X_pooled_mean_b1 = _help(data=train_data_X, num_bins=num_bins1)
    val_data_X_pooled_mean_b1   = _help(data=val_data_X,   num_bins=num_bins1)
    test_data_X_pooled_mean_b1  = _help(data=test_data_X,  num_bins=num_bins1)

    # Use second binning configuration (num_bins2) on training, validation, and test data
    train_data_X_pooled_mean_b2 = _help(data=train_data_X, num_bins=num_bins2)
    val_data_X_pooled_mean_b2   = _help(data=val_data_X,   num_bins=num_bins2)
    test_data_X_pooled_mean_b2  = _help(data=test_data_X,  num_bins=num_bins2)

    return (train_data_X_pooled_mean_b1,
            val_data_X_pooled_mean_b1  ,
            test_data_X_pooled_mean_b1 ,
            train_data_X_pooled_mean_b2,
            val_data_X_pooled_mean_b2  ,
            test_data_X_pooled_mean_b2 )


# Example execution:
# Training, validation, and test data contain length-900 signals along axis 1.
# Generate two pooled representations with 6 and 10 bins.
train_data_X_900_pooled_mean_6,  \
val_data_X_900_pooled_mean_6,    \
test_data_X_900_pooled_mean_6,   \
train_data_X_900_pooled_mean_10, \
val_data_X_900_pooled_mean_10,   \
test_data_X_900_pooled_mean_10   = bin_data(train_data_X_900, val_data_X_900, test_data_X_900, 6, 10)

In [4]:
def add_awgn(x: np.ndarray, snr_db: float = None, noise_std: float = None) -> np.ndarray:
    """
    Additive White Gaussian Noise (AWGN) channel for complex-valued signals.

    Parameters
    ----------
    x : np.ndarray
        Input complex-valued signal. Can be 1D (L,) or 2D (N, L);
        noise is added element-wise with the same SNR definition.
    snr_db : float, optional
        Desired signal-to-noise ratio in dB. If provided, noise_std is computed accordingly.
    noise_std : float, optional
        Standard deviation of the noise (per real component). If provided, snr_db is ignored.

    Returns
    -------
    np.ndarray
        Noisy signal: x + n, where n is complex AWGN.
    """

    # If SNR in dB is given, compute the corresponding noise standard deviation.
    if snr_db is not None:
        # Average signal power over all samples: E[|x|^2]
        signal_power = np.mean(np.abs(x) ** 2)

        # Convert SNR from dB to linear scale
        snr_linear   = 10 ** (snr_db / 10)

        # Noise power from SNR definition: SNR = signal_power / noise_power
        noise_power  = signal_power / snr_linear

        # For complex noise with independent real and imaginary parts:
        # total noise power = 2 * noise_std^2  →  noise_std^2 = noise_power / 2
        noise_std    = np.sqrt(noise_power / 2)

    # If neither snr_db nor noise_std is effectively set, fail explicitly.
    if noise_std is None:
        raise ValueError("Either snr_db or noise_std must be provided.")

    # Generate complex Gaussian noise with zero mean and variance noise_std^2 per real component.
    # np.random.randn(*x.shape) draws real-valued N(0,1) samples with same shape as x.
    noise             = noise_std * (np.random.randn(*x.shape) + 1j * np.random.randn(*x.shape))

    # Return noisy signal
    return x + noise


# Generate independent AWGN realizations for the training,
# validation, and test sets at a nominal SNR of 2 dB.
# The signal power, and therefore the noise standard deviation,
# is computed separately for each dataset.

train_data_X_900_awgn_2db = add_awgn(
    x=train_data_X_900,
    snr_db=2
)

val_data_X_900_awgn_2db = add_awgn(
    x=val_data_X_900,
    snr_db=2
)

test_data_X_900_awgn_2db = add_awgn(
    x=test_data_X_900,
    snr_db=2
)



# Binning: apply the same binning/feature extraction to the noisy data
(train_data_X_900_pooled_mean_6_awgn_2db,
 val_data_X_900_pooled_mean_6_awgn_2db,
 test_data_X_900_pooled_mean_6_awgn_2db,
 train_data_X_900_pooled_mean_10_awgn_2db,
 val_data_X_900_pooled_mean_10_awgn_2db,
 test_data_X_900_pooled_mean_10_awgn_2db,
) = bin_data(train_data_X_900_awgn_2db,
             val_data_X_900_awgn_2db,
             test_data_X_900_awgn_2db,
             num_bins1 = 6             ,
             num_bins2 = 10            )

In [5]:
def _row_power(x):
    """
    Compute average power (E[|x|^2]) per row.

    Parameters
    ----------
    x : array-like, shape (N,) or (B, N)
        Real or complex-valued signal(s).

    Returns
    -------
    np.ndarray
        If x is 1D: shape (1,), containing the mean power of that vector.
        If x is 2D: shape (B,), containing the mean power of each row.
    """
    x = np.asarray(x)

    # Single vector: return a length-1 array with its power
    if x.ndim == 1:
        return np.array([np.mean(np.abs(x) ** 2)], dtype=float)  # shape (1,)

    # Matrix: compute mean power along axis=1 (per row)
    if x.ndim == 2:
        return np.mean(np.abs(x) ** 2, axis=1)                   # shape (rows,)

    # Any other shape is not supported
    raise ValueError("x must be 1D or 2D (N,) or (B,N).")


def add_colored_ar1_noise_snr(x, snr_db, rho=0.9, rng=None):
    """
    Add complex AR(1) colored Gaussian noise, power-matched to a target SNR per row.

    The AR(1) model is:
        n[t] = rho * n[t-1] + e[t],
    where e[t] ~ CN(0, sigma_e^2). For a stationary process,
        Var(n) = noise_power,
    which implies:
        sigma_e^2 = noise_power * (1 - rho^2).

    Parameters
    ----------
    x : np.ndarray, shape (N,) or (B, N)
        Input complex-valued signal(s). If 2D, each row is treated
        as an independent signal for SNR matching.
    snr_db : float
        Desired SNR in dB, defined per row as:
            SNR = Px / Pn
        where Px is the row signal power and Pn is the noise power.
    rho : float, optional
        AR(1) correlation coefficient in [0, 1). Larger rho implies stronger
        temporal correlation (more "colored" noise). Default is 0.9.
    rng : int, np.random.Generator, or None, optional
        Seed or Generator for reproducible noise. If None, a new Generator
        with system entropy is created.

    Returns
    -------
    np.ndarray
        Noisy signal(s), same shape as x.
    """

    # Create a new RNG (np.random.Generator). If `rng` is:
    #   - None: use fresh entropy (non-reproducible)
    #   - int: use it as a seed
    #   - Generator: default_rng will treat it as such
    rng = np.random.default_rng(rng)

    x   = np.asarray(x)
    N   = x.shape[-1]  # number of samples per row

    # Compute per-row signal power: shape (1,) for 1D, (B,) for 2D
    Px   = _row_power(x)

    # Per-row noise power from SNR definition:
    #    SNR_linear = Px / Pn  =>  Pn = Px / SNR_linear
    noise_pow    = Px / (10 ** (snr_db / 10))

    # Output array for the noisy signal (used only for 2D case)
    out          = np.empty_like(x)

    # Precompute factor (1 - rho^2) used in sigma_e^2
    one_minus_r2 = 1.0 - (rho ** 2)

    def draw_row(npow):
        """
        Draw one AR(1) complex noise sequence of length N with target power `npow`.
        """
        # Innovation variance sigma_e^2 chosen such that the AR(1) process
        # has stationary variance equal to `npow`.
        sigma_e2 = max(npow * one_minus_r2, 0.0)

        # Complex Gaussian innovations, zero-mean, variance sigma_e2:
        # Real and imaginary parts each have variance sigma_e2/2.
        e = np.sqrt(sigma_e2 / 2) * (
            rng.standard_normal(N) + 1j * rng.standard_normal(N)
        )

        # Allocate noise sequence
        n = np.zeros(N, dtype=np.complex128)

        # AR(1) recursion. Note: n[0] stays at 0 here; noise "ramps up"
        # over time. For long sequences this transient is negligible.
        for t in range(1, N):
            n[t] = rho * n[t - 1] + e[t]

        return n

    # 1D: single row/vector
    if x.ndim == 1:
        return x + draw_row(noise_pow[0])

    # 2D: process each row independently
    for i in range(x.shape[0]):
        out[i] = x[i] + draw_row(noise_pow[i])

    return out


# Generate independent colored AR(1) Gaussian noise realizations
# at 2 dB SNR for training, validation, and test data
train_data_X_900_car1gn_2db = add_colored_ar1_noise_snr(
    x=train_data_X_900, snr_db=2, rho=0.9, rng=None)

val_data_X_900_car1gn_2db = add_colored_ar1_noise_snr(
    x=val_data_X_900, snr_db=2, rho=0.9, rng=None)

test_data_X_900_car1gn_2db = add_colored_ar1_noise_snr(
    x=test_data_X_900, snr_db=2, rho=0.9, rng=None)

# Binning: extract pooled/normalized features from the noisy
# training, validation, and test sequences
(train_data_X_900_pooled_mean_6_car1gn_2db,
 val_data_X_900_pooled_mean_6_car1gn_2db,
 test_data_X_900_pooled_mean_6_car1gn_2db,
 train_data_X_900_pooled_mean_10_car1gn_2db,
 val_data_X_900_pooled_mean_10_car1gn_2db,
 test_data_X_900_pooled_mean_10_car1gn_2db,
) = bin_data(train_data_X_900_car1gn_2db,
             val_data_X_900_car1gn_2db,
             test_data_X_900_car1gn_2db,
             num_bins1 = 6             ,
             num_bins2 = 10            )

In [ ]:
def save_data_noiseless(train_data_X,
                        train_data_X_pooled_mean_6,
                        train_data_X_pooled_mean_10,
                        train_data_y,
                        val_data_X,
                        val_data_X_pooled_mean_6,
                        val_data_X_pooled_mean_10,
                        val_data_y,
                        test_data_X,
                        test_data_X_pooled_mean_6,
                        test_data_X_pooled_mean_10,
                        test_data_y,
                        n,
                        offset):
    """
    Save noiseless training, validation, and test datasets and their
    pooled-mean features to .npy files.

    Files are written into ../Data/ with a naming convention that encodes:
    - whether the data is Train, Val, or Test,
    - the signal length n,
    - the feature type (raw / pooled_mean_6 / pooled_mean_10),
    - and the Offset flag/value.

    Parameters
    ----------
    train_data_X : np.ndarray
        Raw training signals, shape (N_train, L).
    train_data_X_pooled_mean_6 : np.ndarray
        Training features with 6-bin pooled mean, shape (N_train, 6).
    train_data_X_pooled_mean_10 : np.ndarray
        Training features with 10-bin pooled mean, shape (N_train, 10).
    train_data_y : np.ndarray
        Training labels, shape (N_train,).

    val_data_X : np.ndarray
        Raw validation signals, shape (N_val, L).
    val_data_X_pooled_mean_6 : np.ndarray
        Validation features with 6-bin pooled mean, shape (N_val, 6).
    val_data_X_pooled_mean_10 : np.ndarray
        Validation features with 10-bin pooled mean, shape (N_val, 10).
    val_data_y : np.ndarray
        Validation labels, shape (N_val,).

    test_data_X : np.ndarray
        Raw test signals, shape (N_test, L).
    test_data_X_pooled_mean_6 : np.ndarray
        Test features with 6-bin pooled mean, shape (N_test, 6).
    test_data_X_pooled_mean_10 : np.ndarray
        Test features with 10-bin pooled mean, shape (N_test, 10).
    test_data_y : np.ndarray
        Test labels, shape (N_test,).

    n : int or str
        Signal length used as an identifier in filenames (e.g., 900).
    offset : bool or str
        Offset configuration tag appended to filenames. If not already a
        string, it will be converted (e.g., True -> "True").
    """

    # Convert n and offset to strings for use in filenames
    n_str          = str(n)
    offset_str     = str(offset)

    # Save training arrays
    np.save(
        "../Data/X_Train_" + n_str + "_raw_Offset" + offset_str + ".npy",
        train_data_X,
    )
    np.save(
        "../Data/X_Train_" + n_str + "_pooled_mean_6_Offset" + offset_str + ".npy",
        train_data_X_pooled_mean_6,
    )
    np.save(
        "../Data/X_Train_" + n_str + "_pooled_mean_10_Offset" + offset_str + ".npy",
        train_data_X_pooled_mean_10,
    )
    np.save(
        "../Data/y_Train_" + n_str + "_Offset" + offset_str + ".npy",
        train_data_y,
    )

    # Save validation arrays
    np.save(
        "../Data/X_Validation_" + n_str + "_raw_Offset" + offset_str + ".npy",
        val_data_X,
    )
    np.save(
        "../Data/X_Validation_" + n_str + "_pooled_mean_6_Offset" + offset_str + ".npy",
        val_data_X_pooled_mean_6,
    )
    np.save(
        "../Data/X_Validation_" + n_str + "_pooled_mean_10_Offset" + offset_str + ".npy",
        val_data_X_pooled_mean_10,
    )
    np.save(
        "../Data/y_Validation_" + n_str + "_Offset" + offset_str + ".npy",
        val_data_y,
    )

    # Save test arrays
    np.save(
        "../Data/X_Test_" + n_str + "_raw_Offset" + offset_str + ".npy",
        test_data_X,
    )
    np.save(
        "../Data/X_Test_" + n_str + "_pooled_mean_6_Offset" + offset_str + ".npy",
        test_data_X_pooled_mean_6,
    )
    np.save(
        "../Data/X_Test_" + n_str + "_pooled_mean_10_Offset" + offset_str + ".npy",
        test_data_X_pooled_mean_10,
    )
    np.save(
        "../Data/y_Test_" + n_str + "_Offset" + offset_str + ".npy",
        test_data_y,
    )


# Example execution for the noiseless dataset with n = 900
save_data_noiseless(
    train_data_X                = train_data_X_900,
    train_data_X_pooled_mean_6  = train_data_X_900_pooled_mean_6,
    train_data_X_pooled_mean_10 = train_data_X_900_pooled_mean_10,
    train_data_y                = train_data_y_900,
    val_data_X                  = val_data_X_900,
    val_data_X_pooled_mean_6    = val_data_X_900_pooled_mean_6,
    val_data_X_pooled_mean_10   = val_data_X_900_pooled_mean_10,
    val_data_y                  = val_data_y_900,
    test_data_X                 = test_data_X_900,
    test_data_X_pooled_mean_6   = test_data_X_900_pooled_mean_6,
    test_data_X_pooled_mean_10  = test_data_X_900_pooled_mean_10,
    test_data_y                 = test_data_y_900,
    n                           = 900,
    offset                      = offset,
)

In [ ]:
def save_data_with_noise(
    train_data_X_pooled_mean_6,
    train_data_X_pooled_mean_10,
    val_data_X_pooled_mean_6,
    val_data_X_pooled_mean_10,
    test_data_X_pooled_mean_6,
    test_data_X_pooled_mean_10,
    n,
    offset,
    noise,
    db,
    ):

    """
    Save binned (pooled-mean) noisy features for training, validation,
    and test sets to .npy files.

    The files are saved under ../Data/ with a naming convention that encodes:
    - whether the data is Train, Val, or Test,
    - the signal length n,
    - the feature type (pooled_mean_6 or pooled_mean_10),
    - the offset value,
    - the noise type (e.g., "awgn", "car1gn"),
    - and the SNR in dB.

    Parameters
    ----------
    train_data_X_pooled_mean_6 : np.ndarray
        Training features using 6-bin pooling, shape (N_train, 6).
    train_data_X_pooled_mean_10 : np.ndarray
        Training features using 10-bin pooling, shape (N_train, 10).
    val_data_X_pooled_mean_6 : np.ndarray
        Validation features using 6-bin pooling, shape (N_val, 6).
    val_data_X_pooled_mean_10 : np.ndarray
        Validation features using 10-bin pooling, shape (N_val, 10).
    test_data_X_pooled_mean_6 : np.ndarray
        Test features using 6-bin pooling, shape (N_test, 6).
    test_data_X_pooled_mean_10 : np.ndarray
        Test features using 10-bin pooling, shape (N_test, 10).
    n : int or str
        Signal length identifier (e.g., 900) included in filenames.
    offset : bool or str
        Offset configuration tag included in filenames.
    noise : str
        Noise type tag (e.g., "awgn", "car1gn").
    db : int or float or str
        SNR value in dB included in filenames.
    """

    # Convert values to strings for filename construction
    n_str          = str(n)
    offset_str     = str(offset)
    noise_str      = str(noise)
    db_str         = str(db)

    # Save training arrays
    np.save(
        "../Data/X_Train_"
        + n_str
        + "_pooled_mean_6_Offset"
        + offset_str
        + "_"
        + noise_str
        + "_db"
        + db_str
        + ".npy",
        train_data_X_pooled_mean_6,
    )

    np.save(
        "../Data/X_Train_"
        + n_str
        + "_pooled_mean_10_Offset"
        + offset_str
        + "_"
        + noise_str
        + "_db"
        + db_str
        + ".npy",
        train_data_X_pooled_mean_10,
    )


    # Save validation arrays
    np.save(
        "../Data/X_Validation_"
        + n_str
        + "_pooled_mean_6_Offset"
        + offset_str
        + "_"
        + noise_str
        + "_db"
        + db_str
        + ".npy",
        val_data_X_pooled_mean_6,
    )

    np.save(
        "../Data/X_Validation_"
        + n_str
        + "_pooled_mean_10_Offset"
        + offset_str
        + "_"
        + noise_str
        + "_db"
        + db_str
        + ".npy",
        val_data_X_pooled_mean_10,
    )


    # Save test arrays
    np.save(
        "../Data/X_Test_"
        + n_str
        + "_pooled_mean_6_Offset"
        + offset_str
        + "_"
        + noise_str
        + "_db"
        + db_str
        + ".npy",
        test_data_X_pooled_mean_6,
    )

    np.save(
        "../Data/X_Test_"
        + n_str
        + "_pooled_mean_10_Offset"
        + offset_str
        + "_"
        + noise_str
        + "_db"
        + db_str
        + ".npy",
        test_data_X_pooled_mean_10,
    )


# save AWGN, 2 dB, n = 900
save_data_with_noise(
    train_data_X_pooled_mean_6=train_data_X_900_pooled_mean_6_awgn_2db,
    train_data_X_pooled_mean_10=train_data_X_900_pooled_mean_10_awgn_2db,
    val_data_X_pooled_mean_6=val_data_X_900_pooled_mean_6_awgn_2db,
    val_data_X_pooled_mean_10=val_data_X_900_pooled_mean_10_awgn_2db,
    test_data_X_pooled_mean_6=test_data_X_900_pooled_mean_6_awgn_2db,
    test_data_X_pooled_mean_10=test_data_X_900_pooled_mean_10_awgn_2db,
    n=900,
    offset=False,
    noise="awgn",
    db=2,
)

# save Colored AR(1) noise, 2 dB, n = 900
save_data_with_noise(
    train_data_X_pooled_mean_6=train_data_X_900_pooled_mean_6_car1gn_2db,
    train_data_X_pooled_mean_10=train_data_X_900_pooled_mean_10_car1gn_2db,
    val_data_X_pooled_mean_6=val_data_X_900_pooled_mean_6_car1gn_2db,
    val_data_X_pooled_mean_10=val_data_X_900_pooled_mean_10_car1gn_2db,
    test_data_X_pooled_mean_6=test_data_X_900_pooled_mean_6_car1gn_2db,
    test_data_X_pooled_mean_10=test_data_X_900_pooled_mean_10_car1gn_2db,
    n=900,
    offset=False,
    noise="car1gn",
    db=2,
)